# MPS Gate Optimization Notebook (`pepsy.MpsOptimizer`)

This notebook follows a docs-style progression (similar to Quimb tutorials):

1. Configure backend and contraction optimizer
2. Build a reproducible gate program
3. Run stateful replay loops (DMRG/SVD)
4. Compare DMRG, SVD, and exact application from the same initial state

The comparison metric for final wavefunctions is `core.tn_fidelity`.


In [1]:
# Imports
import quimb.tensor as qtn
import torch

import pepsy as py
core = py.core


## 1) Backend and Contraction Setup

Pick array backend for tensor data, then build the contraction-path optimizer used by local DMRG fitting.


In [2]:
# Backend for tensor arrays
# to_backend = core.backend_cupy(device=0, dtype="complex128")
to_backend = core.backend_torch(dtype=torch.complex128)

# Path optimizer passed into MpsOptimizer(..., opt=optimizer)
optimizer = core.build_optimizer(progbar=False, directory="cash/", parallel=False)


## 2) Build a 1D Gate Program from a 2D Geometry

We start from a 2D square lattice, map sites to 1D indices, then build a mixed gate stream of:
- one-qubit gates (`rx`, `u3`)
- two-qubit gates (`su4`)

The resulting list is the queue consumed by `MpsOptimizer.run(...)`.


In [3]:
# 2D lattice setup
Lx, Ly, cyclic = 4, 4, True
L = Lx * Ly
edges = qtn.edges_2d_square(Lx=Lx, Ly=Ly, cyclic=cyclic)
sites = sorted({(site,) for edge in edges for site in edge})

# Gate parameters
field_h, dt = 0.5, 0.25
rx = py.rx(to_backend(-field_h * dt))
u3 = py.u3(to_backend([-field_h * dt, 0.2, 0.4]))

# 2-qubit gate (SU4 example)
su4_params = to_backend([0.4 * (i + 1) for i in range(15)])
su4 = py.su4(su4_params)

# 2D -> 1D site map
site_map = {(i, j): i * Ly + j for i in range(Lx) for j in range(Ly)}

# Build gate list as (where, gate) tuples
gates = []
for site in sites:
    (xy,) = site
    gates.append(((site_map[xy],), rx))
    gates.append(((site_map[xy],), u3))

for a, b in edges:
    gates.append(((site_map[a], site_map[b]), su4))

print("Total gates:", len(gates))
print("Two-qubit gates:", sum(1 for where, _ in gates if len(where) == 2))


Total gates: 64
Two-qubit gates: 32


## 3) Stateful Replay Pattern

`MpsOptimizer` is stateful:
- repeated `run(...)` calls continue from the current `self.p`
- canonicalization metadata (`info_c["cur_orthog"]`) keeps updating
- `set_gates(...)` replaces queue, `add_gates(...)` appends


In [4]:
# Initial MPS state
chi = 16
p0 = qtn.MPS_rand_state(L=L, bond_dim=2, seed=42)
p0.apply_to_arrays(to_backend)

# Constructor without gates, then set queue explicitly
mps_replay = py.MpsOptimizer(p0.copy(), chi, mode="svd", opt=optimizer)
mps_replay.set_gates(gates)

# Alternate DMRG and SVD over the same current gate list
for step in range(4):
    if step % 2 == 0:
        mps_replay.run(progbar=True, mode="dmrg", n_iter=10)
    else:
        mps_replay.run(progbar=True, mode="svd", fidelity_samples=10)

    print(
        f"step={step + 1}",
        "cur_orthog=", mps_replay.info_c.get("cur_orthog"),
        "F_last=", mps_replay.get_fidelities()[-1],
        "max_bond=", mps_replay.p.max_bond(),
    )


dmrg:   0%|          | 0/64 [00:00<?, ?it/s]

dmrg:   0%|          | 0/64 [00:00<?, ?it/s, 2q=0, ~F=1, bnd=16]

dmrg:   2%|▏         | 1/64 [00:00<00:00, 707.30it/s, 2q=0, ~F=1, bnd=16]

dmrg:   3%|▎         | 2/64 [00:00<00:00, 1186.17it/s, 2q=0, ~F=1, bnd=16]

dmrg:   5%|▍         | 3/64 [00:00<00:00, 1518.39it/s, 2q=0, ~F=1, bnd=16]

...

svd:  95%|█████████▌| 61/64 [00:00<00:00, 508.08it/s, 2q=30, ~F=0.00613, bnd=16]

svd:  97%|█████████▋| 62/64 [00:00<00:00, 508.08it/s, 2q=31, ~F=0.00613, bnd=16]

svd:  98%|█████████▊| 63/64 [00:00<00:00, 508.08it/s, 2q=32, ~F=0.00613, bnd=16]

svd: 100%|██████████| 64/64 [00:00<00:00, 478.28it/s, 2q=32, ~F=0.00613, bnd=16]

step=4 cur_orthog= (14, 14) F_last= 0.006125425799538675 max_bond= 16


## 4) Mode Comparison from the Same Initial State

Now compare three execution modes on the same initial `p0` and same gate program:
- `dmrg` (local fitting)
- `svd` (direct nonlocal gate compression)
- `exact` (exact gate application)


In [5]:
# Optionally deepen the circuit for clearer separation between modes
gates_cmp = list(gates) + list(gates)
chi_cmp = 64

mps_dmrg_cmp = py.MpsOptimizer(p0.copy(), gates_cmp, chi_cmp, mode="dmrg", opt=optimizer)
mps_dmrg_cmp.run(n_iter=10, progbar=True, k_2q_batch=5)

mps_svd_cmp = py.MpsOptimizer(p0.copy(), gates_cmp, chi_cmp, mode="svd", opt=optimizer)
mps_svd_cmp.run(progbar=True, fidelity_samples=10)

mps_exact_cmp = py.MpsOptimizer(p0.copy(), gates_cmp, chi_cmp, mode="exact", opt=optimizer)
mps_exact_cmp.run(progbar=True, fidelity_samples=10)

psi_dmrg = mps_dmrg_cmp.p
psi_svd = mps_svd_cmp.p
psi_exact = mps_exact_cmp.p

print("F(dmrg, exact) =", core.tn_fidelity(psi_dmrg, psi_exact))
print("F(svd,  exact) =", core.tn_fidelity(psi_svd, psi_exact))
print("F(dmrg, svd)   =", core.tn_fidelity(psi_dmrg, psi_svd))


dmrg:   0%|          | 0/128 [00:00<?, ?it/s]

dmrg:   0%|          | 0/128 [00:00<?, ?it/s, 2q=0, ~F=1, bnd=64]

dmrg:   1%|          | 1/128 [00:00<00:00, 2605.16it/s, 2q=0, ~F=1, bnd=64]

dmrg:   2%|▏         | 2/128 [00:00<00:00, 3114.97it/s, 2q=0, ~F=1, bnd=64]

dmrg:   2%|▏         | 3/128 [00:00<00:00, 3090.11it/s, 2q=0, ~F=1, bnd=64]

...

exact:  95%|█████████▌| 61/64 [00:00<00:00, 871.04it/s, 2q=62, ~F=1, bnd=None]

exact:  97%|█████████▋| 62/64 [00:00<00:00, 869.97it/s, 2q=63, ~F=1, bnd=None]

exact:  98%|█████████▊| 63/64 [00:00<00:00, 869.08it/s, 2q=64, ~F=1, bnd=None]

exact: 100%|██████████| 64/64 [00:00<00:00, 880.34it/s, 2q=64, ~F=1, bnd=None]

F(dmrg, exact) = 0.3060994133767222
F(svd,  exact) = 0.18004730451712173
F(dmrg, svd)   = 0.38049759565197283


## 5) Practical API Notes

- `set_gates(new_gates)` replaces the full queue.
- `add_gates(extra_gates)` appends to the existing queue.
- `run(...)` always iterates over the full current queue.
- `k_2q_batch` only affects `mode="dmrg"`.
- `fidelity_samples` controls how often SVD/exact sample the fidelity proxy (`p.norm()`).
